## 容器、buffer 和训练模式

`ModuleList`/`ModuleDict` 会注册子模块；`register_buffer` 保存随设备移动但不参与优化的 Tensor。`train()` 与 `eval()` 会改变 Dropout 和 BatchNorm 的行为。


In [ ]:
import torch
from torch import nn
class ContainerDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 4), nn.ReLU()])
        self.lookup = nn.ModuleDict({'head': nn.Linear(4, 2)})
        self.register_buffer('scale', torch.ones(1))
    def forward(self, x):
        return self.lookup['head'](self.layers[1](self.layers[0](x))) * self.scale

demo = ContainerDemo()
assert 'scale' in demo.state_dict() and any('layers.0' in n for n, _ in demo.named_parameters())
dropout = nn.Dropout(0.5)
dropout.train(); train_output = dropout(torch.ones(100))
dropout.eval(); eval_output = dropout(torch.ones(100))
assert not torch.equal(train_output, eval_output)
print('registered parameters and buffer:', list(demo.state_dict()))

# nn.Module、参数与 logits

## 学习目标

能够定义模块、检查参数注册、追踪前向形状，并正确理解分类 logits。


## 概念模型与执行路径

`nn.Module` 管理子模块、参数、设备移动和训练模式。调用 `model(x)` 会执行 hooks 后进入 `forward`。分类头输出未归一化 logits，交叉熵内部完成 log-softmax。


### 实验 1


In [1]:
import torch
from torch import nn

class Classifier(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8, classes=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, classes))

    def forward(self, inputs):
        return self.net(inputs)

model = Classifier()
print(model)


Classifier(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=3, bias=True)
  )
)


### 实验 2


In [ ]:
batch = torch.randn(5, 4)
logits = model(batch)
print("input -> logits:", batch.shape, "->", logits.shape)
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape), parameter.requires_grad)


### 实验 3


In [ ]:
labels = torch.tensor([0, 2, 1, 0, 2])
loss = nn.CrossEntropyLoss()(logits, labels)
probabilities = logits.softmax(dim=1)
print("loss:", loss.item())
print("probability row sums:", probabilities.sum(dim=1))


### 实验 4


In [ ]:
activations = {}
handle = model.net[0].register_forward_hook(lambda module, args, output: activations.update(linear=output.detach()))
_ = model(batch)
handle.remove()
print("captured hidden shape:", activations["linear"].shape)


## 底层机制

只有赋值为 Module 属性的 `Parameter` 才会被注册。把层放进普通 Python list 会导致参数不出现在 `model.parameters()` 中，应使用 `ModuleList` 或 `Sequential`。


## 检查点

计算该模型参数量，并说明第一层权重为什么是 `(8, 4)` 而不是 `(4, 8)`。


## 试一试

增加一个隐藏层并用 hook 记录每层输出形状；确认最终 logits 仍为 `(5, 3)`。


## 常见错误与调试

手动 softmax 后再用交叉熵、在 `forward` 中临时创建带参数层、忘记调用 `super().__init__()`、标签 dtype 不是 long。
